# AgriPulse Mandi Market Analytics — Week 7 Gold KPI Build

**Notebook:** `05_gold_aggregations.ipynb`  
**Project:** AgriPulse Mandi Market Analytics  
**Layer:** Trusted Silver → Gold  
**Objective:** Define and build 8 dashboard-ready Gold KPI tables and validate their grains, measures, joins, and rerun behaviour.

## Week 7 boundary

This notebook:
- reads **only Trusted Silver** outputs from Week 6;
- documents the 8 approved KPI families from the AgriPulse project playbook;
- builds 8 Gold KPI tables;
- validates key/grain safety, scope, measures and reconciliation;
- records the streaming KPI as **NOT CONFIGURED** until an approved trusted event table exists.

This notebook does **not** rebuild Bronze/Silver/DQ logic and does not build Power BI pages.


## 1. Approved AgriPulse KPI register

The project playbook specifies these eight KPI families:

| # | KPI table | KPI | Proposed Gold grain |
|---|---|---|---|
| 1 | `gold_trusted_observation_count` | Trusted observation count | market + report date |
| 2 | `gold_average_modal_price` | Average modal price | commodity + variety + month |
| 3 | `gold_total_arrival_tonnes` | Total arrival tonnes | commodity + variety + month |
| 4 | `gold_average_price_spread` | Average price spread | market + commodity + variety + month |
| 5 | `gold_market_coverage_rate` | Market coverage rate | report date |
| 6 | `gold_price_change_pct` | Price change % | market + commodity + variety + date |
| 7 | `gold_volatility_index` | Volatility index | market + commodity + variety + month |
| 8 | `gold_fresh_trusted_event_rate` | Fresh trusted event rate | report date |

### KPI formula guards

1. **Trusted observation count:** count distinct approved full-grain observations.
2. **Average modal price:** `SUM(modal_price) / trusted priced observations`.
3. **Total arrival tonnes:** `SUM(arrival_quantity * approved conversion factor)`.
4. **Average price spread:** `AVG(maximum_price - minimum_price)`.
5. **Market coverage rate:** `markets with trusted observations / active eligible markets`.
6. **Price change %:** `(current - previous) / previous`, with zero/null guard.
7. **Volatility index:** approved standard-deviation/variation calculation at a fixed grain/window. This notebook uses standard deviation of trusted modal prices at market + commodity + variety + month.
8. **Fresh trusted event rate:** trusted events within the approved freshness threshold / processed valid events.

**Important:** The formulas above come from the approved project playbook. Where the playbook leaves a configuration open (for example the freshness threshold), the notebook uses a clearly marked configuration variable rather than inventing an undocumented business rule.


In [0]:
# Databricks setup
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

CATALOG = "workspace"
SCHEMA = "default"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print("Active catalog:", spark.catalog.currentCatalog())
print("Active schema:", spark.catalog.currentDatabase())


Active catalog: workspace
Active schema: default


## 2. Week 6 handoff — confirm Trusted Silver exists

Week 7 must consume the validated Trusted outputs. If these tables do not exist, return to Week 6 rather than recreating DQ logic here.


In [0]:
# Required Week 6 Trusted Silver tables
required_tables = [
    "silver_markets_trusted",
    "silver_commodities_trusted",
    "silver_prices_trusted",
    "silver_arrivals_trusted",
]

handoff = []
for t in required_tables:
    exists = spark.catalog.tableExists(t)
    handoff.append((t, exists))

display(spark.createDataFrame(handoff, ["table_name", "exists"]))


table_name,exists
silver_markets_trusted,true
silver_commodities_trusted,true
silver_prices_trusted,true
silver_arrivals_trusted,true


In [0]:
# Stop early if a required Trusted table is missing
missing = [t for t, exists in handoff if not exists]
if missing:
    raise RuntimeError(
        "Week 6 handoff incomplete. Missing Trusted Silver tables: " + ", ".join(missing)
    )

print("PASS: all required Trusted Silver tables exist.")


PASS: all required Trusted Silver tables exist.


## 3. Week 6 reconciliation

The mandatory control is:

`Candidate rows = Trusted rows + Quarantine rows`

Expected variance is zero for markets, commodities, prices and arrivals.

This is only a handoff check; this notebook does not modify Week 6 tables.


In [0]:
# Week 6 Candidate = Trusted + Quarantine reconciliation
reconciliation_sql = '''
WITH reconciliation AS (
    SELECT
        'markets' AS entity,
        (SELECT COUNT(*) FROM silver_markets_candidate) AS candidate_rows,
        (SELECT COUNT(*) FROM silver_markets_trusted) AS trusted_rows,
        (SELECT COUNT(*) FROM quarantine_markets) AS quarantine_rows

    UNION ALL

    SELECT
        'commodities',
        (SELECT COUNT(*) FROM silver_commodities_candidate),
        (SELECT COUNT(*) FROM silver_commodities_trusted),
        (SELECT COUNT(*) FROM quarantine_commodities)

    UNION ALL

    SELECT
        'prices',
        (SELECT COUNT(*) FROM silver_prices_candidate),
        (SELECT COUNT(*) FROM silver_prices_trusted),
        (SELECT COUNT(*) FROM quarantine_prices)

    UNION ALL

    SELECT
        'arrivals',
        (SELECT COUNT(*) FROM silver_arrivals_candidate),
        (SELECT COUNT(*) FROM silver_arrivals_trusted),
        (SELECT COUNT(*) FROM quarantine_arrivals)
)
SELECT
    *,
    candidate_rows - trusted_rows - quarantine_rows AS variance,
    CASE
        WHEN candidate_rows = trusted_rows + quarantine_rows THEN 'PASS'
        ELSE 'CHECK'
    END AS reconciliation_status
FROM reconciliation
ORDER BY entity
'''
display(spark.sql(reconciliation_sql))


entity,candidate_rows,trusted_rows,quarantine_rows,variance,reconciliation_status
arrivals,88220,87137,1083,0,PASS
commodities,164,156,8,0,PASS
markets,606,594,12,0,PASS
prices,96125,94908,1217,0,PASS


> **Evidence:** capture the actual Databricks output of the reconciliation above as part of Week 7 evidence. Do not fabricate PASS results.


## 4. Inspect Trusted Silver schemas and counts


In [0]:
for table_name in required_tables:
    print(f"\n===== {table_name} =====")
    spark.table(table_name).printSchema()
    print("Rows:", spark.table(table_name).count())



===== silver_markets_trusted =====
root
 |-- market_record_id: string (nullable = true)
 |-- market_id: string (nullable = true)
 |-- market_name: string (nullable = true)
 |-- raw_market_name: string (nullable = true)
 |-- market_type: string (nullable = true)
 |-- district_id: string (nullable = true)
 |-- district_name: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- state_name: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- active_flag: boolean (nullable = true)
 |-- valid_from: date (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_record_version: integer (nullable = true)
 |-- source_row_number: integer (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- obs_id_check: string (nullable = true)
 |-- obs_duplicate_check: string (nullable = true)
 |-- ref_check: string (nullable = true)
 |-- dq_status: string 

In [0]:
# Small, dashboard-focused previews
display(spark.table("silver_prices_trusted").select(
    "price_record_id", "market_id", "commodity_id", "variety_id",
    "report_date", "minimum_price", "modal_price", "maximum_price", "price_unit"
).limit(10))

display(spark.table("silver_arrivals_trusted").select(
    "arrival_record_id", "market_id", "commodity_id", "variety_id",
    "report_date", "arrival_quantity", "arrival_unit"
).limit(10))

display(spark.table("silver_markets_trusted").select(
    "market_id", "market_name", "district_name", "state_name", "active_flag"
).limit(10))

display(spark.table("silver_commodities_trusted").select(
    "commodity_id", "commodity_name", "category",
    "variety_id", "variety_name", "arrival_unit", "arrival_to_tonne_factor"
).limit(10))


price_record_id,market_id,commodity_id,variety_id,report_date,minimum_price,modal_price,maximum_price,price_unit
PR0000001,MKT0354,CMD012,VAR0046,2025-04-18,2894.31,3113.43,3307.82,INR_PER_QUINTAL
PR0000002,MKT0268,CMD037,VAR0148,2025-05-14,9630.67,10151.93,10821.8,INR_PER_QUINTAL
PR0000003,MKT0265,CMD012,VAR0048,2025-01-15,4006.73,4045.46,4100.55,INR_PER_QUINTAL
PR0000004,MKT0061,CMD001,VAR0004,2025-03-03,3054.92,3217.33,3385.57,INR_PER_QUINTAL
PR0000005,MKT0406,CMD014,VAR0055,2025-03-19,3123.25,3260.95,3407.57,INR_PER_QUINTAL
PR0000006,MKT0034,CMD029,VAR0114,2025-03-04,8913.76,9427.49,9920.36,INR_PER_QUINTAL
PR0000007,MKT0383,CMD028,VAR0109,2025-05-21,11676.7,12531.04,13211.78,INR_PER_QUINTAL
PR0000008,MKT0394,CMD011,VAR0043,2025-02-03,2613.04,2769.67,2906.68,INR_PER_QUINTAL
PR0000009,MKT0071,CMD036,VAR0141,2025-01-29,2165.97,2260.0,2351.61,INR_PER_QUINTAL
PR0000010,MKT0107,CMD023,VAR0090,2025-01-06,6744.06,6974.92,7162.39,INR_PER_QUINTAL


arrival_record_id,market_id,commodity_id,variety_id,report_date,arrival_quantity,arrival_unit
AR0000001,MKT0489,CMD003,VAR0012,2025-04-15,184.72,TONNE
AR0000002,MKT0139,CMD020,VAR0080,2025-02-11,472.294,QUINTAL
AR0000003,MKT0325,CMD040,VAR0157,2025-06-29,403.579,TONNE
AR0000004,MKT0005,CMD011,VAR0043,2025-04-21,60.794,TONNE
AR0000006,MKT0353,CMD038,VAR0150,2025-02-26,50.368,TONNE
AR0000007,MKT0592,CMD007,VAR0026,2025-03-26,2500.562,QUINTAL
AR0000008,MKT0257,CMD039,VAR0154,2025-03-11,18.995,TONNE
AR0000009,MKT0174,CMD018,VAR0069,2025-05-31,27.222,TONNE
AR0000010,MKT0317,CMD033,VAR0130,2025-04-20,95.596,TONNE
AR0000011,MKT0020,CMD038,VAR0151,2025-06-03,86.729,TONNE


market_id,market_name,district_name,state_name,active_flag
MKT0001,ADILABAD MANDI 01,ADILABAD,TELANGANA,true
MKT0002,ADILABAD MANDI 02,ADILABAD,TELANGANA,true
MKT0003,ADILABAD MANDI 03,ADILABAD,TELANGANA,true
MKT0004,ADILABAD MANDI 04,ADILABAD,TELANGANA,true
MKT0005,ADILABAD MANDI 05,ADILABAD,TELANGANA,true
MKT0006,ADILABAD MANDI 06,ADILABAD,TELANGANA,true
MKT0007,ADILABAD MANDI 07,ADILABAD,TELANGANA,true
MKT0008,ADILABAD MANDI 08,ADILABAD,TELANGANA,true
MKT0009,ADILABAD MANDI 09,ADILABAD,TELANGANA,true
MKT0010,ADILABAD MANDI 10,ADILABAD,TELANGANA,true


commodity_id,commodity_name,category,variety_id,variety_name,arrival_unit,arrival_to_tonne_factor
CMD001,TOMATO,VEGETABLES,VAR0001,STANDARD,TONNE,1.0
CMD001,TOMATO,VEGETABLES,VAR0002,GRADE A,QUINTAL,0.1
CMD001,TOMATO,VEGETABLES,VAR0003,LOCAL,TONNE,1.0
CMD001,TOMATO,VEGETABLES,VAR0004,PREMIUM,TONNE,1.0
CMD002,ONION,VEGETABLES,VAR0005,STANDARD,QUINTAL,0.1
CMD002,ONION,VEGETABLES,VAR0006,GRADE A,TONNE,1.0
CMD002,ONION,VEGETABLES,VAR0007,LOCAL,TONNE,1.0
CMD003,POTATO,VEGETABLES,VAR0009,STANDARD,TONNE,1.0
CMD003,POTATO,VEGETABLES,VAR0010,GRADE A,TONNE,1.0
CMD003,POTATO,VEGETABLES,VAR0011,LOCAL,QUINTAL,0.1


## 5. Shared grain and join safety checks

The approved daily business grain is:

`market_id + commodity_id + variety_id + report_date`

Price/arrival joins must use all four keys. Commodity/variety lookup joins must use `commodity_id + variety_id`.

Do not use text names as join keys.


In [0]:
# Price full-grain uniqueness
price_dupes = spark.sql('''
SELECT market_id, commodity_id, variety_id, report_date, COUNT(*) AS occurrences
FROM silver_prices_trusted
GROUP BY market_id, commodity_id, variety_id, report_date
HAVING COUNT(*) > 1
''')
display(price_dupes)

# Arrival full-grain uniqueness
arrival_dupes = spark.sql('''
SELECT market_id, commodity_id, variety_id, report_date, COUNT(*) AS occurrences
FROM silver_arrivals_trusted
GROUP BY market_id, commodity_id, variety_id, report_date
HAVING COUNT(*) > 1
''')
display(arrival_dupes)


market_id,commodity_id,variety_id,report_date,occurrences


market_id,commodity_id,variety_id,report_date,occurrences


In [0]:
# Lookup uniqueness
market_dupes = spark.sql('''
SELECT market_id, COUNT(*) AS occurrences
FROM silver_markets_trusted
GROUP BY market_id
HAVING COUNT(*) > 1
''')

commodity_dupes = spark.sql('''
SELECT commodity_id, variety_id, COUNT(*) AS occurrences
FROM silver_commodities_trusted
GROUP BY commodity_id, variety_id
HAVING COUNT(*) > 1
''')

display(market_dupes)
display(commodity_dupes)


market_id,occurrences


commodity_id,variety_id,occurrences


## 6. Build reusable trusted input views

Only fields required by the KPI contracts are selected. This keeps Gold logic explicit and avoids carrying unnecessary Silver columns.


In [0]:
spark.sql('''
CREATE OR REPLACE TEMP VIEW trusted_price_base AS
SELECT
    price_record_id,
    market_id,
    commodity_id,
    variety_id,
    CAST(report_date AS DATE) AS report_date,
    CAST(minimum_price AS DECIMAL(18,2)) AS minimum_price,
    CAST(modal_price AS DECIMAL(18,2)) AS modal_price,
    CAST(maximum_price AS DECIMAL(18,2)) AS maximum_price,
    price_unit
FROM silver_prices_trusted
WHERE report_date IS NOT NULL
''')

spark.sql('''
CREATE OR REPLACE TEMP VIEW trusted_arrival_base AS
SELECT
    arrival_record_id,
    market_id,
    commodity_id,
    variety_id,
    CAST(report_date AS DATE) AS report_date,
    CAST(arrival_quantity AS DOUBLE) AS arrival_quantity,
    arrival_unit
FROM silver_arrivals_trusted
WHERE report_date IS NOT NULL
''')

spark.sql('''
CREATE OR REPLACE TEMP VIEW trusted_market_dim AS
SELECT
    market_id,
    market_name,
    district_id,
    district_name,
    state_code,
    state_name,
    active_flag
FROM silver_markets_trusted
''')

spark.sql('''
CREATE OR REPLACE TEMP VIEW trusted_commodity_dim AS
SELECT
    commodity_id,
    commodity_name,
    category,
    variety_id,
    variety_name,
    arrival_unit,
    CAST(arrival_to_tonne_factor AS DOUBLE) AS arrival_to_tonne_factor
FROM silver_commodities_trusted
''')

print("Trusted input views created.")


Trusted input views created.


# KPI 1 — Trusted Observation Count

**Table:** `gold_trusted_observation_count`

**Question:** How many distinct trusted price observations are available for each market and reporting date?

**Formula:** `COUNT(DISTINCT price_record_id)`

**Grain:** market + report date

**Scope:** Trusted price observations with a usable report date.

**Source:** `silver_prices_trusted` + market dimension.


In [0]:
spark.sql('''
CREATE OR REPLACE TABLE gold_trusted_observation_count
USING DELTA
AS
SELECT
    p.market_id,
    m.market_name,
    m.district_name,
    m.state_name,
    p.report_date,
    COUNT(DISTINCT p.price_record_id) AS trusted_observation_count,
    current_timestamp() AS _gold_created_at,
    'agripulse_gold_v1.0' AS _gold_schema_version
FROM trusted_price_base p
LEFT JOIN trusted_market_dim m
    ON p.market_id = m.market_id
GROUP BY
    p.market_id,
    m.market_name,
    m.district_name,
    m.state_name,
    p.report_date
''')

display(spark.sql("SELECT * FROM gold_trusted_observation_count ORDER BY report_date, market_id LIMIT 20"))


market_id,market_name,district_name,state_name,report_date,trusted_observation_count,_gold_created_at,_gold_schema_version
MKT0003,ADILABAD MANDI 03,ADILABAD,TELANGANA,2025-01-01,2,2026-09-05T09:59:53.123Z,agripulse_gold_v1.0
MKT0006,ADILABAD MANDI 06,ADILABAD,TELANGANA,2025-01-01,2,2026-09-05T09:59:53.123Z,agripulse_gold_v1.0
MKT0007,ADILABAD MANDI 07,ADILABAD,TELANGANA,2025-01-01,1,2026-09-05T09:59:53.123Z,agripulse_gold_v1.0
MKT0008,ADILABAD MANDI 08,ADILABAD,TELANGANA,2025-01-01,1,2026-09-05T09:59:53.123Z,agripulse_gold_v1.0
MKT0009,ADILABAD MANDI 09,ADILABAD,TELANGANA,2025-01-01,1,2026-09-05T09:59:53.123Z,agripulse_gold_v1.0
MKT0010,ADILABAD MANDI 10,ADILABAD,TELANGANA,2025-01-01,1,2026-09-05T09:59:53.123Z,agripulse_gold_v1.0
MKT0012,KARIMNAGAR MANDI 02,KARIMNAGAR,TELANGANA,2025-01-01,3,2026-09-05T09:59:53.123Z,agripulse_gold_v1.0
MKT0013,KARIMNAGAR MANDI 03,KARIMNAGAR,TELANGANA,2025-01-01,1,2026-09-05T09:59:53.123Z,agripulse_gold_v1.0
MKT0014,KARIMNAGAR MANDI 04,KARIMNAGAR,TELANGANA,2025-01-01,2,2026-09-05T09:59:53.123Z,agripulse_gold_v1.0
MKT0016,KARIMNAGAR MANDI 06,KARIMNAGAR,TELANGANA,2025-01-01,1,2026-09-05T09:59:53.123Z,agripulse_gold_v1.0


In [0]:
# KPI 1 validation
display(spark.sql('''
SELECT
    SUM(CASE WHEN market_id IS NULL THEN 1 ELSE 0 END) AS null_market_keys,
    SUM(CASE WHEN report_date IS NULL THEN 1 ELSE 0 END) AS null_date_keys,
    SUM(CASE WHEN trusted_observation_count < 0 THEN 1 ELSE 0 END) AS negative_measures
FROM gold_trusted_observation_count
'''))

display(spark.sql('''
SELECT market_id, report_date, COUNT(*) AS occurrences
FROM gold_trusted_observation_count
GROUP BY market_id, report_date
HAVING COUNT(*) > 1
'''))

display(spark.sql('''
WITH expected AS (
    SELECT COUNT(DISTINCT price_record_id) AS trusted_price_observations
    FROM trusted_price_base
),
gold AS (
    SELECT COALESCE(SUM(trusted_observation_count), 0) AS gold_observations
    FROM gold_trusted_observation_count
)
SELECT *,
       CASE WHEN trusted_price_observations = gold_observations
            THEN 'PASS' ELSE 'CHECK' END AS reconciliation_status
FROM expected CROSS JOIN gold
'''))


null_market_keys,null_date_keys,negative_measures
0,0,0


market_id,report_date,occurrences


trusted_price_observations,gold_observations,reconciliation_status
94908,94908,PASS


# KPI 2 — Average Modal Price

**Table:** `gold_average_modal_price`

**Question:** What is the average trusted modal price for each commodity/variety by month?

**Formula:** `SUM(modal_price) / trusted priced observations`

**Grain:** commodity + variety + month

**Scope:** Trusted price observations with non-null modal price and report date.

**Source:** `silver_prices_trusted` + commodity dimension.


In [0]:
spark.sql('''
CREATE OR REPLACE TABLE gold_average_modal_price
USING DELTA
AS
SELECT
    p.commodity_id,
    c.commodity_name,
    c.category,
    p.variety_id,
    c.variety_name,
    YEAR(p.report_date) AS year,
    MONTH(p.report_date) AS month,
    SUM(CAST(p.modal_price AS DOUBLE)) AS modal_price_sum,
    COUNT(p.modal_price) AS trusted_priced_observations,
    SUM(CAST(p.modal_price AS DOUBLE))
        / NULLIF(COUNT(p.modal_price), 0) AS average_modal_price,
    current_timestamp() AS _gold_created_at,
    'agripulse_gold_v1.0' AS _gold_schema_version
FROM trusted_price_base p
LEFT JOIN trusted_commodity_dim c
    ON p.commodity_id = c.commodity_id
   AND p.variety_id = c.variety_id
WHERE p.modal_price IS NOT NULL
GROUP BY
    p.commodity_id, c.commodity_name, c.category,
    p.variety_id, c.variety_name,
    YEAR(p.report_date), MONTH(p.report_date)
''')

display(spark.sql("SELECT * FROM gold_average_modal_price ORDER BY year, month, commodity_id LIMIT 20"))


commodity_id,commodity_name,category,variety_id,variety_name,year,month,modal_price_sum,trusted_priced_observations,average_modal_price,_gold_created_at,_gold_schema_version
CMD001,TOMATO,VEGETABLES,VAR0002,GRADE A,2025,1,266261.07,109,2442.762110091743,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD001,TOMATO,VEGETABLES,VAR0003,LOCAL,2025,1,284154.44,109,2606.9214678899084,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD001,TOMATO,VEGETABLES,VAR0004,PREMIUM,2025,1,292619.37999999995,103,2840.9648543689314,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD001,TOMATO,VEGETABLES,VAR0001,STANDARD,2025,1,260921.05999999997,115,2268.8787826086955,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD001,null,null,VAR0060,null,2025,1,10568.41,1,10568.41,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD002,ONION,VEGETABLES,VAR0005,STANDARD,2025,1,162962.76999999996,89,1831.0423595505613,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD002,null,null,VAR0008,null,2025,1,170200.42999999996,75,2269.339066666666,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD002,null,null,VAR0083,null,2025,1,6371.47,1,6371.47,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD002,ONION,VEGETABLES,VAR0007,LOCAL,2025,1,232199.21000000005,110,2110.9019090909096,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD002,null,null,VAR0069,null,2025,1,7115.29,1,7115.29,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0


In [0]:
# KPI 2 validation
display(spark.sql('''
SELECT
    SUM(CASE WHEN commodity_id IS NULL THEN 1 ELSE 0 END) AS null_commodity_keys,
    SUM(CASE WHEN variety_id IS NULL THEN 1 ELSE 0 END) AS null_variety_keys,
    SUM(CASE WHEN trusted_priced_observations <= 0 THEN 1 ELSE 0 END) AS invalid_counts,
    SUM(CASE WHEN average_modal_price IS NULL THEN 1 ELSE 0 END) AS null_average
FROM gold_average_modal_price
'''))

display(spark.sql('''
SELECT commodity_id, variety_id, year, month, COUNT(*) AS occurrences
FROM gold_average_modal_price
GROUP BY commodity_id, variety_id, year, month
HAVING COUNT(*) > 1
'''))

display(spark.sql('''
WITH expected AS (
    SELECT
        SUM(CAST(modal_price AS DOUBLE)) AS expected_sum,
        COUNT(modal_price) AS expected_count
    FROM trusted_price_base
    WHERE modal_price IS NOT NULL
),
gold AS (
    SELECT
        SUM(modal_price_sum) AS gold_sum,
        SUM(trusted_priced_observations) AS gold_count
    FROM gold_average_modal_price
)
SELECT *,
       CASE WHEN ABS(expected_sum - gold_sum) < 0.01
                 AND expected_count = gold_count
            THEN 'PASS' ELSE 'CHECK' END AS reconciliation_status
FROM expected CROSS JOIN gold
'''))


null_commodity_keys,null_variety_keys,invalid_counts,null_average
0,0,0,0


commodity_id,variety_id,year,month,occurrences


expected_sum,expected_count,gold_sum,gold_count,reconciliation_status
5.863729972000033E8,94908,5.863729971999999E8,94908,PASS


# KPI 3 — Total Arrival Tonnes

**Table:** `gold_total_arrival_tonnes`

**Question:** How much trusted arrival volume was received, expressed in tonnes?

**Formula:** `SUM(arrival_quantity * approved arrival_to_tonne_factor)`

**Grain:** commodity + variety + month

**Scope:** Trusted arrivals with usable date, non-negative quantity and an approved conversion factor.

**Source:** `silver_arrivals_trusted` + commodity dimension.

The factor comes from the approved `arrival_to_tonne_factor` field in the AgriPulse commodity dictionary.


In [0]:
spark.sql('''
CREATE OR REPLACE TABLE gold_total_arrival_tonnes
USING DELTA
AS
SELECT
    a.commodity_id,
    c.commodity_name,
    c.category,
    a.variety_id,
    c.variety_name,
    YEAR(a.report_date) AS year,
    MONTH(a.report_date) AS month,
    SUM(
        CAST(a.arrival_quantity AS DOUBLE)
        * CAST(c.arrival_to_tonne_factor AS DOUBLE)
    ) AS total_arrival_tonnes,
    COUNT(DISTINCT a.arrival_record_id) AS trusted_arrival_observations,
    current_timestamp() AS _gold_created_at,
    'agripulse_gold_v1.0' AS _gold_schema_version
FROM trusted_arrival_base a
INNER JOIN trusted_commodity_dim c
    ON a.commodity_id = c.commodity_id
   AND a.variety_id = c.variety_id
WHERE a.arrival_quantity IS NOT NULL
  AND a.arrival_quantity >= 0
  AND c.arrival_to_tonne_factor IS NOT NULL
GROUP BY
    a.commodity_id, c.commodity_name, c.category,
    a.variety_id, c.variety_name,
    YEAR(a.report_date), MONTH(a.report_date)
''')

display(spark.sql("SELECT * FROM gold_total_arrival_tonnes ORDER BY year, month, commodity_id LIMIT 20"))


commodity_id,commodity_name,category,variety_id,variety_name,year,month,total_arrival_tonnes,trusted_arrival_observations,_gold_created_at,_gold_schema_version
CMD001,TOMATO,VEGETABLES,VAR0002,GRADE A,2025,1,10229.811999999998,93,2026-09-05T10:00:35.342Z,agripulse_gold_v1.0
CMD001,TOMATO,VEGETABLES,VAR0003,LOCAL,2025,1,9404.668,100,2026-09-05T10:00:35.342Z,agripulse_gold_v1.0
CMD001,TOMATO,VEGETABLES,VAR0004,PREMIUM,2025,1,7895.042,88,2026-09-05T10:00:35.342Z,agripulse_gold_v1.0
CMD001,TOMATO,VEGETABLES,VAR0001,STANDARD,2025,1,1.0011931011000002E7,113,2026-09-05T10:00:35.342Z,agripulse_gold_v1.0
CMD002,ONION,VEGETABLES,VAR0005,STANDARD,2025,1,8074.0283,88,2026-09-05T10:00:35.342Z,agripulse_gold_v1.0
CMD002,ONION,VEGETABLES,VAR0007,LOCAL,2025,1,10170.826,104,2026-09-05T10:00:35.342Z,agripulse_gold_v1.0
CMD002,ONION,VEGETABLES,VAR0006,GRADE A,2025,1,2.0008399647000004E7,90,2026-09-05T10:00:35.342Z,agripulse_gold_v1.0
CMD003,POTATO,VEGETABLES,VAR0012,PREMIUM,2025,1,8799.601,78,2026-09-05T10:00:35.342Z,agripulse_gold_v1.0
CMD003,POTATO,VEGETABLES,VAR0010,GRADE A,2025,1,8876.218000000004,104,2026-09-05T10:00:35.342Z,agripulse_gold_v1.0
CMD003,POTATO,VEGETABLES,VAR0009,STANDARD,2025,1,10122.990999999998,87,2026-09-05T10:00:35.342Z,agripulse_gold_v1.0


In [0]:
# KPI 3 validation
display(spark.sql('''
SELECT
    SUM(CASE WHEN commodity_id IS NULL THEN 1 ELSE 0 END) AS null_commodity_keys,
    SUM(CASE WHEN variety_id IS NULL THEN 1 ELSE 0 END) AS null_variety_keys,
    SUM(CASE WHEN total_arrival_tonnes < 0 THEN 1 ELSE 0 END) AS negative_tonne_totals
FROM gold_total_arrival_tonnes
'''))

display(spark.sql('''
WITH expected AS (
    SELECT
        SUM(
            CAST(a.arrival_quantity AS DOUBLE)
            * CAST(c.arrival_to_tonne_factor AS DOUBLE)
        ) AS expected_tonnes,
        COUNT(DISTINCT a.arrival_record_id) AS expected_arrival_observations
    FROM trusted_arrival_base a
    INNER JOIN trusted_commodity_dim c
        ON a.commodity_id = c.commodity_id
       AND a.variety_id = c.variety_id
    WHERE a.arrival_quantity IS NOT NULL
      AND a.arrival_quantity >= 0
      AND c.arrival_to_tonne_factor IS NOT NULL
),
gold AS (
    SELECT
        SUM(total_arrival_tonnes) AS gold_tonnes,
        SUM(trusted_arrival_observations) AS gold_arrival_observations
    FROM gold_total_arrival_tonnes
)
SELECT *,
       CASE WHEN ABS(expected_tonnes - gold_tonnes) < 0.01
                 AND expected_arrival_observations = gold_arrival_observations
            THEN 'PASS' ELSE 'CHECK' END AS reconciliation_status
FROM expected CROSS JOIN gold
'''))


null_commodity_keys,null_variety_keys,negative_tonne_totals
0,0,0


expected_tonnes,expected_arrival_observations,gold_tonnes,gold_arrival_observations,reconciliation_status
1.3875830967164886E9,84741,1.3875830967164977E9,84741,PASS


# KPI 4 — Average Price Spread

**Table:** `gold_average_price_spread`

**Question:** What is the average difference between maximum and minimum trusted price?

**Formula:** `AVG(maximum_price - minimum_price)`

**Grain:** market + commodity + variety + month

**Scope:** Trusted price observations with all three price measures present.

**Source:** `silver_prices_trusted`.


In [0]:
spark.sql('''
CREATE OR REPLACE TABLE gold_average_price_spread
USING DELTA
AS
SELECT
    p.market_id,
    m.market_name,
    m.state_name,
    p.commodity_id,
    c.commodity_name,
    p.variety_id,
    c.variety_name,
    YEAR(p.report_date) AS year,
    MONTH(p.report_date) AS month,
    AVG(
        CAST(p.maximum_price AS DOUBLE)
        - CAST(p.minimum_price AS DOUBLE)
    ) AS average_price_spread,
    COUNT(*) AS trusted_priced_observations,
    current_timestamp() AS _gold_created_at,
    'agripulse_gold_v1.0' AS _gold_schema_version
FROM trusted_price_base p
LEFT JOIN trusted_market_dim m
    ON p.market_id = m.market_id
LEFT JOIN trusted_commodity_dim c
    ON p.commodity_id = c.commodity_id
   AND p.variety_id = c.variety_id
WHERE p.minimum_price IS NOT NULL
  AND p.modal_price IS NOT NULL
  AND p.maximum_price IS NOT NULL
GROUP BY
    p.market_id, m.market_name, m.state_name,
    p.commodity_id, c.commodity_name,
    p.variety_id, c.variety_name,
    YEAR(p.report_date), MONTH(p.report_date)
''')

display(spark.sql("SELECT * FROM gold_average_price_spread ORDER BY year, month, market_id LIMIT 20"))


market_id,market_name,state_name,commodity_id,commodity_name,variety_id,variety_name,year,month,average_price_spread,trusted_priced_observations,_gold_created_at,_gold_schema_version
MKT0001,ADILABAD MANDI 01,TELANGANA,CMD031,RED CHILLI,VAR0123,LOCAL,2025,1,1115.33,1,2026-09-05T10:00:58.778Z,agripulse_gold_v1.0
MKT0001,ADILABAD MANDI 01,TELANGANA,CMD004,BRINJAL,VAR0016,PREMIUM,2025,1,109.35000000000036,1,2026-09-05T10:00:58.778Z,agripulse_gold_v1.0
MKT0001,ADILABAD MANDI 01,TELANGANA,CMD033,BANANA,VAR0132,PREMIUM,2025,1,208.5999999999999,1,2026-09-05T10:00:58.778Z,agripulse_gold_v1.0
MKT0001,ADILABAD MANDI 01,TELANGANA,CMD005,CABBAGE,VAR0017,STANDARD,2025,1,87.70000000000005,1,2026-09-05T10:00:58.778Z,agripulse_gold_v1.0
MKT0001,ADILABAD MANDI 01,TELANGANA,CMD002,null,VAR0008,null,2025,1,96.30999999999995,1,2026-09-05T10:00:58.778Z,agripulse_gold_v1.0
MKT0001,ADILABAD MANDI 01,TELANGANA,CMD016,null,VAR0016,null,2025,1,182.46000000000004,1,2026-09-05T10:00:58.778Z,agripulse_gold_v1.0
MKT0001,ADILABAD MANDI 01,TELANGANA,CMD001,TOMATO,VAR0004,PREMIUM,2025,1,121.89000000000033,1,2026-09-05T10:00:58.778Z,agripulse_gold_v1.0
MKT0001,ADILABAD MANDI 01,TELANGANA,CMD019,MASOOR DAL,VAR0074,GRADE A,2025,1,655.0100000000002,1,2026-09-05T10:00:58.778Z,agripulse_gold_v1.0
MKT0001,ADILABAD MANDI 01,TELANGANA,CMD035,ORANGE,VAR0137,STANDARD,2025,1,448.0799999999999,1,2026-09-05T10:00:58.778Z,agripulse_gold_v1.0
MKT0001,ADILABAD MANDI 01,TELANGANA,CMD030,CUMIN SEED,VAR0119,LOCAL,2025,1,2075.1800000000003,1,2026-09-05T10:00:58.778Z,agripulse_gold_v1.0


In [0]:
# KPI 4 validation
display(spark.sql('''
SELECT
    SUM(CASE WHEN market_id IS NULL THEN 1 ELSE 0 END) AS null_market_keys,
    SUM(CASE WHEN commodity_id IS NULL THEN 1 ELSE 0 END) AS null_commodity_keys,
    SUM(CASE WHEN variety_id IS NULL THEN 1 ELSE 0 END) AS null_variety_keys,
    SUM(CASE WHEN average_price_spread < 0 THEN 1 ELSE 0 END) AS negative_spreads
FROM gold_average_price_spread
'''))

display(spark.sql('''
SELECT market_id, commodity_id, variety_id, year, month, COUNT(*) AS occurrences
FROM gold_average_price_spread
GROUP BY market_id, commodity_id, variety_id, year, month
HAVING COUNT(*) > 1
'''))

display(spark.sql('''
WITH expected AS (
    SELECT
        AVG(CAST(maximum_price AS DOUBLE) - CAST(minimum_price AS DOUBLE)) AS expected_overall_spread
    FROM trusted_price_base
    WHERE minimum_price IS NOT NULL
      AND modal_price IS NOT NULL
      AND maximum_price IS NOT NULL
),
gold AS (
    SELECT
        SUM(average_price_spread * trusted_priced_observations)
        / NULLIF(SUM(trusted_priced_observations), 0) AS gold_overall_spread
    FROM gold_average_price_spread
)
SELECT *,
       CASE WHEN ABS(expected_overall_spread - gold_overall_spread) < 0.01
            THEN 'PASS' ELSE 'CHECK' END AS reconciliation_status
FROM expected CROSS JOIN gold
'''))


null_market_keys,null_commodity_keys,null_variety_keys,negative_spreads
0,0,0,0


market_id,commodity_id,variety_id,year,month,occurrences


expected_overall_spread,gold_overall_spread,reconciliation_status
17428.65073018082,17428.65073018074,PASS


# KPI 5 — Market Coverage Rate

**Table:** `gold_market_coverage_rate`

**Question:** What share of active eligible markets had at least one trusted price observation on each reporting date?

**Formula:** `markets with trusted observations / active eligible markets`

**Grain:** report date

**Scope:** active markets from `silver_markets_trusted`; numerator is distinct markets represented by trusted price observations.

**Guard:** denominator zero returns NULL.


In [0]:
spark.sql('''
CREATE OR REPLACE TABLE gold_market_coverage_rate
USING DELTA
AS
WITH dates AS (
    SELECT DISTINCT report_date
    FROM trusted_price_base
    WHERE report_date IS NOT NULL
),
active_markets AS (
    SELECT COUNT(DISTINCT market_id) AS active_eligible_markets
    FROM trusted_market_dim
    WHERE active_flag = TRUE
),
observed AS (
    SELECT
        report_date,
        COUNT(DISTINCT market_id) AS markets_with_trusted_observations
    FROM trusted_price_base
    GROUP BY report_date
)
SELECT
    d.report_date,
    COALESCE(o.markets_with_trusted_observations, 0) AS markets_with_trusted_observations,
    a.active_eligible_markets,
    CASE
        WHEN a.active_eligible_markets > 0
        THEN COALESCE(o.markets_with_trusted_observations, 0) * 1.0
             / a.active_eligible_markets
        ELSE NULL
    END AS market_coverage_rate,
    current_timestamp() AS _gold_created_at,
    'agripulse_gold_v1.0' AS _gold_schema_version
FROM dates d
CROSS JOIN active_markets a
LEFT JOIN observed o
    ON d.report_date = o.report_date
''')

display(spark.sql("SELECT * FROM gold_market_coverage_rate ORDER BY report_date LIMIT 20"))


report_date,markets_with_trusted_observations,active_eligible_markets,market_coverage_rate,_gold_created_at,_gold_schema_version
2025-01-01,338,594,0.5690235690235690,2026-09-05T10:01:22.105Z,agripulse_gold_v1.0
2025-01-02,359,594,0.6043771043771044,2026-09-05T10:01:22.105Z,agripulse_gold_v1.0
2025-01-03,368,594,0.6195286195286195,2026-09-05T10:01:22.105Z,agripulse_gold_v1.0
2025-01-04,348,594,0.5858585858585859,2026-09-05T10:01:22.105Z,agripulse_gold_v1.0
2025-01-05,357,594,0.6010101010101010,2026-09-05T10:01:22.105Z,agripulse_gold_v1.0
2025-01-06,333,594,0.5606060606060606,2026-09-05T10:01:22.105Z,agripulse_gold_v1.0
2025-01-07,355,594,0.5976430976430976,2026-09-05T10:01:22.105Z,agripulse_gold_v1.0
2025-01-08,342,594,0.5757575757575758,2026-09-05T10:01:22.105Z,agripulse_gold_v1.0
2025-01-09,314,594,0.5286195286195286,2026-09-05T10:01:22.105Z,agripulse_gold_v1.0
2025-01-10,349,594,0.5875420875420875,2026-09-05T10:01:22.105Z,agripulse_gold_v1.0


In [0]:
# KPI 5 validation
display(spark.sql('''
SELECT
    SUM(CASE WHEN report_date IS NULL THEN 1 ELSE 0 END) AS null_dates,
    SUM(CASE WHEN market_coverage_rate < 0 OR market_coverage_rate > 1
             THEN 1 ELSE 0 END) AS invalid_coverage_rates
FROM gold_market_coverage_rate
'''))

display(spark.sql('''
WITH expected AS (
    SELECT COUNT(DISTINCT market_id) AS active_eligible_markets
    FROM trusted_market_dim
    WHERE active_flag = TRUE
),
gold AS (
    SELECT MAX(active_eligible_markets) AS gold_active_eligible_markets
    FROM gold_market_coverage_rate
)
SELECT *,
       CASE WHEN active_eligible_markets = gold_active_eligible_markets
            THEN 'PASS' ELSE 'CHECK' END AS denominator_reconciliation
FROM expected CROSS JOIN gold
'''))


null_dates,invalid_coverage_rates
0,0


active_eligible_markets,gold_active_eligible_markets,denominator_reconciliation
594,594,PASS


# KPI 6 — Price Change %

**Table:** `gold_price_change_pct`

**Question:** How did the trusted modal price change versus the previous available trusted observation?

**Formula:** `(current - previous) / previous`

**Grain:** market + commodity + variety + date

**Scope:** trusted modal prices with a previous observation in the same market/commodity/variety series.

**Guard:** if previous price is NULL or zero, `price_change_pct` is NULL.


In [0]:
spark.sql('''
CREATE OR REPLACE TEMP VIEW trusted_price_with_lag AS
SELECT
    p.*,
    LAG(modal_price) OVER (
        PARTITION BY market_id, commodity_id, variety_id
        ORDER BY report_date
    ) AS previous_modal_price
FROM trusted_price_base p
WHERE modal_price IS NOT NULL
''')

spark.sql('''
CREATE OR REPLACE TABLE gold_price_change_pct
USING DELTA
AS
SELECT
    p.market_id,
    m.market_name,
    m.state_name,
    p.commodity_id,
    c.commodity_name,
    p.variety_id,
    c.variety_name,
    p.report_date,
    p.modal_price AS current_modal_price,
    p.previous_modal_price,
    CASE
        WHEN p.previous_modal_price IS NULL OR p.previous_modal_price = 0
        THEN NULL
        ELSE
            (CAST(p.modal_price AS DOUBLE) - CAST(p.previous_modal_price AS DOUBLE))
            / CAST(p.previous_modal_price AS DOUBLE)
    END AS price_change_pct,
    current_timestamp() AS _gold_created_at,
    'agripulse_gold_v1.0' AS _gold_schema_version
FROM trusted_price_with_lag p
LEFT JOIN trusted_market_dim m
    ON p.market_id = m.market_id
LEFT JOIN trusted_commodity_dim c
    ON p.commodity_id = c.commodity_id
   AND p.variety_id = c.variety_id
''')

display(spark.sql("SELECT * FROM gold_price_change_pct ORDER BY report_date, market_id LIMIT 20"))


market_id,market_name,state_name,commodity_id,commodity_name,variety_id,variety_name,report_date,current_modal_price,previous_modal_price,price_change_pct,_gold_created_at,_gold_schema_version
MKT0003,ADILABAD MANDI 03,TELANGANA,CMD021,SOYBEAN,VAR0081,STANDARD,2025-01-01,4426.05,null,null,2026-09-05T10:01:47.137Z,agripulse_gold_v1.0
MKT0003,ADILABAD MANDI 03,TELANGANA,CMD027,JUTE,VAR0105,STANDARD,2025-01-01,4547.12,null,null,2026-09-05T10:01:47.137Z,agripulse_gold_v1.0
MKT0006,ADILABAD MANDI 06,TELANGANA,CMD013,BAJRA,VAR0052,PREMIUM,2025-01-01,3372.07,null,null,2026-09-05T10:01:47.137Z,agripulse_gold_v1.0
MKT0006,ADILABAD MANDI 06,TELANGANA,CMD027,JUTE,VAR0106,GRADE A,2025-01-01,6024.60,null,null,2026-09-05T10:01:47.137Z,agripulse_gold_v1.0
MKT0007,ADILABAD MANDI 07,TELANGANA,CMD040,GRAPES,VAR0159,LOCAL,2025-01-01,7785.03,null,null,2026-09-05T10:01:47.137Z,agripulse_gold_v1.0
MKT0008,ADILABAD MANDI 08,TELANGANA,CMD037,APPLE,VAR0146,GRADE A,2025-01-01,11894.19,null,null,2026-09-05T10:01:47.137Z,agripulse_gold_v1.0
MKT0009,ADILABAD MANDI 09,TELANGANA,CMD028,TURMERIC,VAR0111,LOCAL,2025-01-01,14778.64,null,null,2026-09-05T10:01:47.137Z,agripulse_gold_v1.0
MKT0010,ADILABAD MANDI 10,TELANGANA,CMD013,BAJRA,VAR0049,STANDARD,2025-01-01,2278.35,null,null,2026-09-05T10:01:47.137Z,agripulse_gold_v1.0
MKT0012,KARIMNAGAR MANDI 02,TELANGANA,CMD009,RICE,VAR0036,PREMIUM,2025-01-01,3830.22,null,null,2026-09-05T10:01:47.137Z,agripulse_gold_v1.0
MKT0012,KARIMNAGAR MANDI 02,TELANGANA,CMD002,null,VAR0008,null,2025-01-01,2227.60,null,null,2026-09-05T10:01:47.137Z,agripulse_gold_v1.0


In [0]:
# KPI 6 validation
display(spark.sql('''
SELECT
    SUM(CASE WHEN market_id IS NULL THEN 1 ELSE 0 END) AS null_market_keys,
    SUM(CASE WHEN commodity_id IS NULL THEN 1 ELSE 0 END) AS null_commodity_keys,
    SUM(CASE WHEN variety_id IS NULL THEN 1 ELSE 0 END) AS null_variety_keys,
    SUM(CASE WHEN price_change_pct IS NOT NULL
                  AND (price_change_pct = CAST('NaN' AS DOUBLE)
                       OR price_change_pct = CAST('Infinity' AS DOUBLE)
                       OR price_change_pct = CAST('-Infinity' AS DOUBLE))
             THEN 1 ELSE 0 END) AS invalid_percent_values
FROM gold_price_change_pct
'''))

display(spark.sql('''
SELECT market_id, commodity_id, variety_id, report_date, COUNT(*) AS occurrences
FROM gold_price_change_pct
GROUP BY market_id, commodity_id, variety_id, report_date
HAVING COUNT(*) > 1
'''))


null_market_keys,null_commodity_keys,null_variety_keys,invalid_percent_values
0,0,0,0


market_id,commodity_id,variety_id,report_date,occurrences


# KPI 7 — Volatility Index

**Table:** `gold_volatility_index`

**Question:** Which market/commodity/variety combinations show the greatest monthly variation in modal price?

**Formula used here:** standard deviation of trusted modal price at a fixed month grain.

**Grain:** market + commodity + variety + month

**Scope:** trusted modal-price observations with non-null dates and prices.

The approved playbook permits a standard-deviation or variation calculation at a fixed grain/window; the chosen implementation is explicitly recorded here so it is auditable.


In [0]:
spark.sql('''
CREATE OR REPLACE TABLE gold_volatility_index
USING DELTA
AS
SELECT
    p.market_id,
    m.market_name,
    m.state_name,
    p.commodity_id,
    c.commodity_name,
    p.variety_id,
    c.variety_name,
    YEAR(p.report_date) AS year,
    MONTH(p.report_date) AS month,
    STDDEV_SAMP(CAST(p.modal_price AS DOUBLE)) AS volatility_index,
    AVG(CAST(p.modal_price AS DOUBLE)) AS average_modal_price,
    COUNT(p.modal_price) AS trusted_priced_observations,
    current_timestamp() AS _gold_created_at,
    'agripulse_gold_v1.0' AS _gold_schema_version
FROM trusted_price_base p
LEFT JOIN trusted_market_dim m
    ON p.market_id = m.market_id
LEFT JOIN trusted_commodity_dim c
    ON p.commodity_id = c.commodity_id
   AND p.variety_id = c.variety_id
WHERE p.modal_price IS NOT NULL
GROUP BY
    p.market_id, m.market_name, m.state_name,
    p.commodity_id, c.commodity_name,
    p.variety_id, c.variety_name,
    YEAR(p.report_date), MONTH(p.report_date)
''')

display(spark.sql("SELECT * FROM gold_volatility_index ORDER BY year, month, volatility_index DESC LIMIT 20"))


market_id,market_name,state_name,commodity_id,commodity_name,variety_id,variety_name,year,month,volatility_index,average_modal_price,trusted_priced_observations,_gold_created_at,_gold_schema_version
MKT0469,MEERUT MANDI 09,UTTAR PRADESH,CMD030,CUMIN SEED,VAR0119,LOCAL,2025,1,3044.8017997892734,24622.89,2,2026-09-05T10:02:06.942Z,agripulse_gold_v1.0
MKT0250,COIMBATORE MANDI 10,TAMIL NADU,CMD031,RED CHILLI,VAR0124,PREMIUM,2025,1,2559.875040319351,19494.815,2,2026-09-05T10:02:06.942Z,agripulse_gold_v1.0
MKT0199,JALGAON MANDI 09,MAHARASHTRA,CMD031,RED CHILLI,VAR0123,LOCAL,2025,1,1994.8260114731816,15222.265,2,2026-09-05T10:02:06.942Z,agripulse_gold_v1.0
MKT0496,BANASKANTHA MANDI 06,GUJARAT,CMD031,RED CHILLI,VAR0123,LOCAL,2025,1,1915.94825003182,16129.07,2,2026-09-05T10:02:06.942Z,agripulse_gold_v1.0
MKT0064,ANANTAPUR MANDI 04,ANDHRA PRADESH,CMD030,CUMIN SEED,VAR0118,GRADE A,2025,1,1831.6469795787616,23777.85,2,2026-09-05T10:02:06.942Z,agripulse_gold_v1.0
MKT9999,null,null,CMD029,CORIANDER SEED,VAR0114,GRADE A,2025,1,1729.9084559016417,9032.71,2,2026-09-05T10:02:06.942Z,agripulse_gold_v1.0
MKT0598,SAMBALPUR MANDI 08,ODISHA,CMD031,RED CHILLI,VAR0123,LOCAL,2025,1,1523.1716462861302,16682.504999999997,2,2026-09-05T10:02:06.942Z,agripulse_gold_v1.0
MKT0422,AGRA MANDI 02,UTTAR PRADESH,CMD030,CUMIN SEED,VAR0118,GRADE A,2025,1,1460.3664219811412,24335.985,2,2026-09-05T10:02:06.942Z,agripulse_gold_v1.0
MKT0315,DEWAS MANDI 05,MADHYA PRADESH,CMD028,TURMERIC,VAR0111,LOCAL,2025,1,1414.3974101362037,15168.27,2,2026-09-05T10:02:06.942Z,agripulse_gold_v1.0
MKT9999,null,null,CMD029,CORIANDER SEED,VAR0113,STANDARD,2025,1,1393.7428210577445,8670.385,2,2026-09-05T10:02:06.942Z,agripulse_gold_v1.0


In [0]:
# KPI 7 validation
display(spark.sql('''
SELECT
    SUM(CASE WHEN market_id IS NULL THEN 1 ELSE 0 END) AS null_market_keys,
    SUM(CASE WHEN commodity_id IS NULL THEN 1 ELSE 0 END) AS null_commodity_keys,
    SUM(CASE WHEN variety_id IS NULL THEN 1 ELSE 0 END) AS null_variety_keys,
    SUM(CASE WHEN volatility_index < 0 THEN 1 ELSE 0 END) AS negative_volatility
FROM gold_volatility_index
'''))

display(spark.sql('''
SELECT market_id, commodity_id, variety_id, year, month, COUNT(*) AS occurrences
FROM gold_volatility_index
GROUP BY market_id, commodity_id, variety_id, year, month
HAVING COUNT(*) > 1
'''))


null_market_keys,null_commodity_keys,null_variety_keys,negative_volatility
0,0,0,0


market_id,commodity_id,variety_id,year,month,occurrences


# KPI 8 — Fresh Trusted Event Rate

**Table:** `gold_fresh_trusted_event_rate`

**Formula:** `trusted events within freshness threshold / processed valid events`

**Important:** The AgriPulse project playbook introduces the price-update event stream later. This current Week 7 project does not assume a streaming Trusted table exists.

Therefore:
- if an approved trusted event table is configured, this section computes the KPI;
- otherwise it creates the Gold KPI table with a `NOT_CONFIGURED` status and zero rows of KPI observations;
- no fake stream source, event counts, or freshness threshold is invented.

### Configuration

Set `TRUSTED_EVENT_TABLE` only when your team has an approved trusted event table.

Set `FRESHNESS_THRESHOLD_MINUTES` only after the threshold is approved/documented.


In [0]:
# Streaming configuration — intentionally explicit
TRUSTED_EVENT_TABLE = None
FRESHNESS_THRESHOLD_MINUTES = None

if TRUSTED_EVENT_TABLE is not None and FRESHNESS_THRESHOLD_MINUTES is not None:
    if not spark.catalog.tableExists(TRUSTED_EVENT_TABLE):
        raise RuntimeError(
            f"Configured trusted event table does not exist: {TRUSTED_EVENT_TABLE}"
        )

    # Expected event columns must be adapted to the approved Week 10 event schema.
    # Do not invent mappings here. Update only after the team approves the event contract.
    print("Trusted event table configured:", TRUSTED_EVENT_TABLE)
    print("Freshness threshold (minutes):", FRESHNESS_THRESHOLD_MINUTES)
else:
    print(
        "Fresh trusted event rate is NOT_CONFIGURED for Week 7 because "
        "no approved Trusted event table and freshness threshold were supplied."
    )


Fresh trusted event rate is NOT_CONFIGURED for Week 7 because no approved Trusted event table and freshness threshold were supplied.


In [0]:
# Create KPI 8 table safely without inventing a stream source.
if TRUSTED_EVENT_TABLE is None or FRESHNESS_THRESHOLD_MINUTES is None:
    spark.sql('''
    CREATE OR REPLACE TABLE gold_fresh_trusted_event_rate
    USING DELTA
    AS
    SELECT
        CAST(NULL AS DATE) AS report_date,
        CAST(NULL AS BIGINT) AS fresh_trusted_events,
        CAST(NULL AS BIGINT) AS processed_valid_events,
        CAST(NULL AS DOUBLE) AS fresh_trusted_event_rate,
        'NOT_CONFIGURED' AS kpi_status,
        current_timestamp() AS _gold_created_at,
        'agripulse_gold_v1.0' AS _gold_schema_version
    WHERE 1 = 0
    ''')
else:
    raise NotImplementedError(
        "Configure the approved Week 10 event schema and freshness rule before "
        "computing gold_fresh_trusted_event_rate. No event schema is invented in Week 7."
    )

display(spark.sql("SELECT * FROM gold_fresh_trusted_event_rate"))


report_date,fresh_trusted_events,processed_valid_events,fresh_trusted_event_rate,kpi_status,_gold_created_at,_gold_schema_version


# 7. Gold KPI table catalog

The following eight tables are the Week 7 Gold KPI outputs.


In [0]:
gold_tables = [
    "gold_trusted_observation_count",
    "gold_average_modal_price",
    "gold_total_arrival_tonnes",
    "gold_average_price_spread",
    "gold_market_coverage_rate",
    "gold_price_change_pct",
    "gold_volatility_index",
    "gold_fresh_trusted_event_rate",
]

catalog_rows = []
for t in gold_tables:
    row_count = spark.table(t).count()
    catalog_rows.append((t, row_count))

display(spark.createDataFrame(catalog_rows, ["gold_table", "row_count"]))


gold_table,row_count
gold_trusted_observation_count,63308
gold_average_modal_price,1209
gold_total_arrival_tonnes,936
gold_average_price_spread,87791
gold_market_coverage_rate,181
gold_price_change_pct,94908
gold_volatility_index,87791
gold_fresh_trusted_event_rate,0


In [0]:
# Show Gold schemas
for t in gold_tables:
    print(f"\n===== {t} =====")
    spark.table(t).printSchema()



===== gold_trusted_observation_count =====
root
 |-- market_id: string (nullable = true)
 |-- market_name: string (nullable = true)
 |-- district_name: string (nullable = true)
 |-- state_name: string (nullable = true)
 |-- report_date: date (nullable = true)
 |-- trusted_observation_count: long (nullable = true)
 |-- _gold_created_at: timestamp (nullable = true)
 |-- _gold_schema_version: string (nullable = true)


===== gold_average_modal_price =====
root
 |-- commodity_id: string (nullable = true)
 |-- commodity_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- variety_id: string (nullable = true)
 |-- variety_name: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- modal_price_sum: double (nullable = true)
 |-- trusted_priced_observations: long (nullable = true)
 |-- average_modal_price: double (nullable = true)
 |-- _gold_created_at: timestamp (nullable = true)
 |-- _gold_schema_version: string (nulla

# 8. Cross-table Gold validation

This section gives one compact validation scorecard for the Gold layer.


In [0]:
validation_rows = []

def add_validation(name, passed, detail):
    validation_rows.append((name, "PASS" if passed else "CHECK", detail))

# 1-4: table existence
for t in gold_tables:
    add_validation(
        f"{t}_exists",
        spark.catalog.tableExists(t),
        "Gold table exists"
    )

# KPI 1 key uniqueness
k1_dupes = spark.sql('''
SELECT COUNT(*) AS n
FROM (
    SELECT market_id, report_date
    FROM gold_trusted_observation_count
    GROUP BY market_id, report_date
    HAVING COUNT(*) > 1
)
''').first()["n"]
add_validation("KPI1_grain_unique", k1_dupes == 0, f"duplicate groups={k1_dupes}")

# KPI 2 key uniqueness
k2_dupes = spark.sql('''
SELECT COUNT(*) AS n
FROM (
    SELECT commodity_id, variety_id, year, month
    FROM gold_average_modal_price
    GROUP BY commodity_id, variety_id, year, month
    HAVING COUNT(*) > 1
)
''').first()["n"]
add_validation("KPI2_grain_unique", k2_dupes == 0, f"duplicate groups={k2_dupes}")

# KPI 3 key uniqueness
k3_dupes = spark.sql('''
SELECT COUNT(*) AS n
FROM (
    SELECT commodity_id, variety_id, year, month
    FROM gold_total_arrival_tonnes
    GROUP BY commodity_id, variety_id, year, month
    HAVING COUNT(*) > 1
)
''').first()["n"]
add_validation("KPI3_grain_unique", k3_dupes == 0, f"duplicate groups={k3_dupes}")

# KPI 4 key uniqueness
k4_dupes = spark.sql('''
SELECT COUNT(*) AS n
FROM (
    SELECT market_id, commodity_id, variety_id, year, month
    FROM gold_average_price_spread
    GROUP BY market_id, commodity_id, variety_id, year, month
    HAVING COUNT(*) > 1
)
''').first()["n"]
add_validation("KPI4_grain_unique", k4_dupes == 0, f"duplicate groups={k4_dupes}")

# KPI 5 coverage bounds
k5_bad = spark.sql('''
SELECT COUNT(*) AS n
FROM gold_market_coverage_rate
WHERE market_coverage_rate < 0 OR market_coverage_rate > 1
''').first()["n"]
add_validation("KPI5_coverage_bounds", k5_bad == 0, f"invalid rows={k5_bad}")

# KPI 7 volatility bounds
k7_bad = spark.sql('''
SELECT COUNT(*) AS n
FROM gold_volatility_index
WHERE volatility_index < 0
''').first()["n"]
add_validation("KPI7_volatility_nonnegative", k7_bad == 0, f"invalid rows={k7_bad}")

# KPI 8 explicit status
k8_status = "NOT_CONFIGURED" if TRUSTED_EVENT_TABLE is None else "CONFIGURED"
add_validation("KPI8_streaming_status", True, k8_status)

display(spark.createDataFrame(
    validation_rows,
    ["check_name", "status", "detail"]
))


check_name,status,detail
gold_trusted_observation_count_exists,PASS,Gold table exists
gold_average_modal_price_exists,PASS,Gold table exists
gold_total_arrival_tonnes_exists,PASS,Gold table exists
gold_average_price_spread_exists,PASS,Gold table exists
gold_market_coverage_rate_exists,PASS,Gold table exists
gold_price_change_pct_exists,PASS,Gold table exists
gold_volatility_index_exists,PASS,Gold table exists
gold_fresh_trusted_event_rate_exists,PASS,Gold table exists
KPI1_grain_unique,PASS,duplicate groups=0
KPI2_grain_unique,PASS,duplicate groups=0


# 9. Controlled rerun / replay proof

The business output should remain unchanged when the Gold build is repeated. Technical timestamps are allowed to change.

This cell snapshots the business columns of the eight KPI tables, then the user can rerun the Gold build sections and execute the comparison below.


In [0]:
# Create short-lived baselines for business columns
spark.sql('''
CREATE OR REPLACE TABLE gold_kpi_rerun_baseline_1
USING DELTA AS
SELECT market_id, report_date, trusted_observation_count
FROM gold_trusted_observation_count
''')

spark.sql('''
CREATE OR REPLACE TABLE gold_kpi_rerun_baseline_2
USING DELTA AS
SELECT commodity_id, variety_id, year, month,
       trusted_priced_observations, average_modal_price
FROM gold_average_modal_price
''')

spark.sql('''
CREATE OR REPLACE TABLE gold_kpi_rerun_baseline_3
USING DELTA AS
SELECT commodity_id, variety_id, year, month,
       trusted_arrival_observations, total_arrival_tonnes
FROM gold_total_arrival_tonnes
''')

spark.sql('''
CREATE OR REPLACE TABLE gold_kpi_rerun_baseline_4
USING DELTA AS
SELECT market_id, commodity_id, variety_id, year, month,
       trusted_priced_observations, average_price_spread
FROM gold_average_price_spread
''')

spark.sql('''
CREATE OR REPLACE TABLE gold_kpi_rerun_baseline_5
USING DELTA AS
SELECT report_date, markets_with_trusted_observations,
       active_eligible_markets, market_coverage_rate
FROM gold_market_coverage_rate
''')

spark.sql('''
CREATE OR REPLACE TABLE gold_kpi_rerun_baseline_6
USING DELTA AS
SELECT market_id, commodity_id, variety_id, report_date,
       current_modal_price, previous_modal_price, price_change_pct
FROM gold_price_change_pct
''')

spark.sql('''
CREATE OR REPLACE TABLE gold_kpi_rerun_baseline_7
USING DELTA AS
SELECT market_id, commodity_id, variety_id, year, month,
       volatility_index, average_modal_price, trusted_priced_observations
FROM gold_volatility_index
''')

spark.sql('''
CREATE OR REPLACE TABLE gold_kpi_rerun_baseline_8
USING DELTA AS
SELECT report_date, fresh_trusted_events,
       processed_valid_events, fresh_trusted_event_rate, kpi_status
FROM gold_fresh_trusted_event_rate
''')

print("Baseline tables created. Rerun the Gold build sections, then execute the next cell.")


Baseline tables created. Rerun the Gold build sections, then execute the next cell.


In [0]:
# Compare baseline vs current business outputs.
# Any non-zero result requires investigation.
checks = []

comparisons = [
    (1, "gold_kpi_rerun_baseline_1", "gold_trusted_observation_count",
     "market_id, report_date, trusted_observation_count"),
    (2, "gold_kpi_rerun_baseline_2", "gold_average_modal_price",
     "commodity_id, variety_id, year, month, trusted_priced_observations, average_modal_price"),
    (3, "gold_kpi_rerun_baseline_3", "gold_total_arrival_tonnes",
     "commodity_id, variety_id, year, month, trusted_arrival_observations, total_arrival_tonnes"),
    (4, "gold_kpi_rerun_baseline_4", "gold_average_price_spread",
     "market_id, commodity_id, variety_id, year, month, trusted_priced_observations, average_price_spread"),
    (5, "gold_kpi_rerun_baseline_5", "gold_market_coverage_rate",
     "report_date, markets_with_trusted_observations, active_eligible_markets, market_coverage_rate"),
    (6, "gold_kpi_rerun_baseline_6", "gold_price_change_pct",
     "market_id, commodity_id, variety_id, report_date, current_modal_price, previous_modal_price, price_change_pct"),
    (7, "gold_kpi_rerun_baseline_7", "gold_volatility_index",
     "market_id, commodity_id, variety_id, year, month, volatility_index, average_modal_price, trusted_priced_observations"),
    (8, "gold_kpi_rerun_baseline_8", "gold_fresh_trusted_event_rate",
     "report_date, fresh_trusted_events, processed_valid_events, fresh_trusted_event_rate, kpi_status"),
]

for kpi_no, baseline, current, cols in comparisons:
    sql = f'''
    WITH baseline_minus_current AS (
        SELECT {cols} FROM {baseline}
        EXCEPT
        SELECT {cols} FROM {current}
    ),
    current_minus_baseline AS (
        SELECT {cols} FROM {current}
        EXCEPT
        SELECT {cols} FROM {baseline}
    )
    SELECT
        (SELECT COUNT(*) FROM baseline_minus_current)
        + (SELECT COUNT(*) FROM current_minus_baseline) AS changed_rows
    '''
    changed = spark.sql(sql).first()["changed_rows"]
    checks.append((f"KPI{kpi_no}_rerun", "PASS" if changed == 0 else "CHECK", changed))

display(spark.createDataFrame(checks, ["check_name", "status", "changed_rows"]))


check_name,status,changed_rows
KPI1_rerun,PASS,0
KPI2_rerun,PASS,0
KPI3_rerun,PASS,0
KPI4_rerun,PASS,0
KPI5_rerun,PASS,0
KPI6_rerun,PASS,0
KPI7_rerun,PASS,0
KPI8_rerun,PASS,0


In [0]:
# Cleanup short-lived baseline tables
for t in [
    "gold_kpi_rerun_baseline_1",
    "gold_kpi_rerun_baseline_2",
    "gold_kpi_rerun_baseline_3",
    "gold_kpi_rerun_baseline_4",
    "gold_kpi_rerun_baseline_5",
    "gold_kpi_rerun_baseline_6",
    "gold_kpi_rerun_baseline_7",
    "gold_kpi_rerun_baseline_8",
]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

print("Rerun baseline cleanup complete.")


Rerun baseline cleanup complete.


# 10. Evidence checklist

Capture actual Databricks outputs for:

- Week 6 Candidate = Trusted + Quarantine reconciliation.
- Trusted Silver input counts.
- Full-grain uniqueness checks.
- Lookup uniqueness checks.
- Each Gold table schema/sample output.
- KPI validation scorecard.
- Measure reconciliations for KPIs 1–4.
- Market coverage denominator check.
- Controlled rerun comparison.
- KPI 8 configuration status.

Recommended evidence files:

```text
screenshots/
├── week07_gold_metrics.png
└── week07_gold_validation.png
```

Do not use screenshots containing fabricated numbers.


# 11. Gold KPI contract summary

## `gold_trusted_observation_count`
- **Question:** How many distinct trusted observations exist by market/date?
- **Grain:** market + date
- **Measure:** `COUNT(DISTINCT price_record_id)`

## `gold_average_modal_price`
- **Question:** What is the average trusted modal price?
- **Grain:** commodity + variety + month
- **Measure:** `SUM(modal_price) / COUNT(modal_price)`

## `gold_total_arrival_tonnes`
- **Question:** How much arrival volume was received in tonnes?
- **Grain:** commodity + variety + month
- **Measure:** `SUM(arrival_quantity * arrival_to_tonne_factor)`

## `gold_average_price_spread`
- **Question:** What is the average maximum-minus-minimum price spread?
- **Grain:** market + commodity + variety + month
- **Measure:** `AVG(maximum_price - minimum_price)`

## `gold_market_coverage_rate`
- **Question:** What share of active eligible markets have trusted observations?
- **Grain:** date
- **Measure:** observed markets / active eligible markets

## `gold_price_change_pct`
- **Question:** How did modal price change from the previous trusted observation?
- **Grain:** market + commodity + variety + date
- **Measure:** `(current - previous) / previous`

## `gold_volatility_index`
- **Question:** Which market/commodity/variety combinations have the greatest monthly price variation?
- **Grain:** market + commodity + variety + month
- **Measure:** standard deviation of modal price

## `gold_fresh_trusted_event_rate`
- **Question:** What share of valid processed events became trusted within the freshness threshold?
- **Grain:** date
- **Measure:** fresh trusted events / processed valid events
- **Week 7 status:** not configured unless an approved event source and freshness threshold exist.


# 12. Week 7 exit checklist

- [ ] Week 6 reconciliation is PASS.
- [ ] Trusted Silver tables exist.
- [ ] Daily full-grain uniqueness is verified.
- [ ] Market and commodity/variety lookup keys are unique.
- [ ] Eight KPI contracts are documented.
- [ ] Eight Gold KPI tables are created.
- [ ] Gold grains are explicit.
- [ ] KPI measures are validated.
- [ ] Rerun comparison shows zero changed business rows.
- [ ] Evidence screenshots are captured from actual Databricks output.
- [ ] `docs/gold_metrics_definition.md` is updated.
- [ ] `weekly_logs/week07_log.md` is updated.
- [ ] AI Transparency Note is added to the weekly log.
- [ ] GitHub commit/push completed.

## AI Transparency Note

AI assistance was used to help structure SQL, validation logic and documentation. The project team must independently execute the notebook, inspect actual results, verify the approved KPI definitions/configuration, and accept or modify the generated logic. AI-generated text or SQL is not execution evidence.


In [0]:
tables = spark.sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_name LIKE 'gold_%'
    ORDER BY table_name
""")

display(tables)

table_name
gold_average_modal_price
gold_average_price_spread
gold_fresh_trusted_event_rate
gold_market_coverage_rate
gold_price_change_pct
gold_total_arrival_tonnes
gold_trusted_observation_count
gold_volatility_index


In [0]:
df = spark.table("gold_average_modal_price")

display(df.limit(20))

commodity_id,commodity_name,category,variety_id,variety_name,year,month,modal_price_sum,trusted_priced_observations,average_modal_price,_gold_created_at,_gold_schema_version
CMD028,TURMERIC,SPICES,VAR0109,STANDARD,2025,5,1120689.2199999997,100,11206.892199999997,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD013,BAJRA,CEREALS,VAR0049,STANDARD,2025,1,245201.82000000007,97,2527.8538144329905,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD026,SUGARCANE,COMMERCIAL CROPS,VAR0104,PREMIUM,2025,3,49809.64000000003,101,493.1647524752478,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD010,WHEAT,CEREALS,VAR0039,LOCAL,2025,1,365810.40000000014,118,3100.0881355932215,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD001,TOMATO,VEGETABLES,VAR0002,GRADE A,2025,4,210899.68,94,2243.6136170212767,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD007,null,null,VAR0025,null,2025,6,265238.74,103,2575.133398058252,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD033,BANANA,FRUITS,VAR0131,LOCAL,2025,4,327665.0,117,2800.5555555555557,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD007,null,null,VAR0025,null,2025,1,345967.5100000001,120,2883.0625833333343,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD026,SUGARCANE,COMMERCIAL CROPS,VAR0104,PREMIUM,2025,5,41932.33000000001,99,423.558888888889,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0
CMD001,TOMATO,VEGETABLES,VAR0002,GRADE A,2025,5,262084.39999999994,124,2113.5838709677414,2026-09-05T10:00:14.464Z,agripulse_gold_v1.0


In [0]:
from pyspark.sql import functions as F

# ============================================================
# WEEK 7 - ALL 8 GOLD KPI OUTPUT
# ============================================================

kpi1 = spark.table("gold_trusted_observation_count").agg(
    F.count("*").alias("gold_rows"),
    F.round(F.sum("trusted_observation_count"), 2).alias("kpi_value")
).withColumn("KPI", F.lit("1. Trusted Observation Count"))

kpi2 = spark.table("gold_average_modal_price").agg(
    F.count("*").alias("gold_rows"),
    F.round(F.avg("average_modal_price"), 2).alias("kpi_value")
).withColumn("KPI", F.lit("2. Average Modal Price"))

kpi3 = spark.table("gold_total_arrival_tonnes").agg(
    F.count("*").alias("gold_rows"),
    F.round(F.sum("total_arrival_tonnes"), 2).alias("kpi_value")
).withColumn("KPI", F.lit("3. Total Arrival Tonnes"))

kpi4 = spark.table("gold_average_price_spread").agg(
    F.count("*").alias("gold_rows"),
    F.round(F.avg("average_price_spread"), 2).alias("kpi_value")
).withColumn("KPI", F.lit("4. Average Price Spread"))

kpi5 = spark.table("gold_market_coverage_rate").agg(
    F.count("*").alias("gold_rows"),
    F.round(F.avg("market_coverage_rate") * 100, 2).alias("kpi_value")
).withColumn("KPI", F.lit("5. Market Coverage Rate (%)"))

kpi6 = spark.table("gold_price_change_pct").agg(
    F.count("*").alias("gold_rows"),
    F.round(F.avg("price_change_pct") * 100, 2).alias("kpi_value")
).withColumn("KPI", F.lit("6. Price Change (%)"))

kpi7 = spark.table("gold_volatility_index").agg(
    F.count("*").alias("gold_rows"),
    F.round(F.avg("volatility_index"), 2).alias("kpi_value")
).withColumn("KPI", F.lit("7. Volatility Index"))

kpi8 = spark.table("gold_fresh_trusted_event_rate").agg(
    F.count("*").alias("gold_rows")
).withColumn(
    "kpi_value",
    F.lit(None).cast("double")
).withColumn(
    "KPI",
    F.lit("8. Fresh Trusted Event Rate")
)

# Combine all KPIs
all_kpis = (
    kpi1
    .unionByName(kpi2)
    .unionByName(kpi3)
    .unionByName(kpi4)
    .unionByName(kpi5)
    .unionByName(kpi6)
    .unionByName(kpi7)
    .unionByName(kpi8)
    .select("KPI", "gold_rows", "kpi_value")
)

display(all_kpis)

KPI,gold_rows,kpi_value
1. Trusted Observation Count,63308,94908.0
2. Average Modal Price,1209,6099.37
3. Total Arrival Tonnes,936,1.38758309672E9
4. Average Price Spread,87791,17125.15
5. Market Coverage Rate (%),181,58.88
6. Price Change (%),94908,-4.86
7. Volatility Index,87791,201.15
8. Fresh Trusted Event Rate,0,null
